[See docs/DISCLAIMER_SNIPPET.md](../../../DISCLAIMER_SNIPPET.md)

# Alpha-AGI Business v2 • Colab Notebook

This walkthrough runs the actual bounded supplied-input evidence planner. Use an already installed checkout on Python 3.11–3.13. No model, GPU, credential, network service or package installation is required by these cells. Colab deployment itself has not been qualified. The original service remains a separately configured path described in the README.


## 0 · Import the installed planner


In [ ]:
import copy
import hashlib
import json
from pathlib import Path
from alpha_factory_v1.demos.alpha_agi_business_2_v1 import planning
from alpha_factory_v1.demos.alpha_agi_business_3_v1.enterprise import read_json
print("Installed supplied-input planner ready.")


## 1 · Choose explicit example or supplied input


In [ ]:
INPUT_PATH = None  # Set Path("scenario.json") for supplied input.
source = Path(planning.__file__).with_name("example.json") if INPUT_PATH is None else Path(INPUT_PATH)
scenario = copy.deepcopy(read_json(source, planning.MAX_INPUT_BYTES))
print("Explicit synthetic example" if INPUT_PATH is None else "Supplied input")
print(json.dumps(scenario, ensure_ascii=False, indent=2))


## 2 · Edit and compute
Edit scenario values below and rerun. Source labels and value points remain supplied assumptions.


In [ ]:
# Example edit: scenario["policy"]["effortHours"] = 10
def bounded_evaluate(value):
    encoded = json.dumps(value, ensure_ascii=False, separators=(",", ":"), allow_nan=False).encode("utf-8")
    if len(encoded) > planning.MAX_INPUT_BYTES:
        raise ValueError("Input exceeds the 128000-byte bound")
    return planning.evaluate(value)
report = bounded_evaluate(scenario)
print(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False))


## 3 · Inspect and export unsigned evidence
Existing files are never overwritten. Changing inputs requires recomputing and inspecting the report before export.


In [ ]:
if report != bounded_evaluate(scenario):
    raise ValueError("Input or report changed: rerun and inspect the computation before export")
evidence_path = Path("business2-evidence.json")
evidence = {"schema": "agialpha.business2.notebook-evidence.v1", "input": scenario, "report": report}
evidence_bytes = (json.dumps(evidence, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False) + "\n").encode("utf-8")
if len(evidence_bytes) > 2_000_000:
    raise ValueError("Evidence exceeds the 2000000-byte bound")
with evidence_path.open("xb") as target:
    target.write(evidence_bytes)
print(evidence_path, "sha256:" + hashlib.sha256(evidence_bytes).hexdigest())


## 4 · Recover and recompute
Stored report contents are ignored; the supplied input is strictly bounded and recomputed.


In [ ]:
recovered = read_json(evidence_path, 2_000_000)
if not isinstance(recovered, dict) or set(recovered) != {"schema", "input", "report"} or recovered["schema"] != "agialpha.business2.notebook-evidence.v1":
    raise ValueError("Unexpected notebook evidence schema")
recomputed = bounded_evaluate(recovered["input"])
print("Stored report ignored; supplied input recomputed.")
print(json.dumps(recomputed, ensure_ascii=False, indent=2, allow_nan=False))


## 5 · Scope and next steps
This finishes a local computation. No company action, provider work, execution, settlement or payment occurred. Use the dedicated browser workspace for the same bounded workflow.


In [ ]:
# This exact raw input also reopens in the original browser workspace.
raw_input_path = Path("business2-input.json")
raw_input_bytes = (json.dumps(scenario, ensure_ascii=False, separators=(",", ":"), allow_nan=False) + "\n").encode("utf-8")
if len(raw_input_bytes) > planning.MAX_INPUT_BYTES:
    raise ValueError("Raw input exceeds the 128000-byte bound")
with raw_input_path.open("xb") as target:
    target.write(raw_input_bytes)
print(raw_input_path, "sha256:" + hashlib.sha256(raw_input_bytes).hexdigest())
print(planning.SCOPE)
